In [1]:
import pandas as pd

file_path = 'us_road_accidents.csv'
df = pd.read_csv(file_path)

print(df.head())

   Severity  Start_Lat  Start_Lng  Distance(mi)    Timezone  Temperature(F)  \
0         2  39.928060 -82.831184          0.01  US/Eastern            37.9   
1         2  39.063150 -84.032610          0.01  US/Eastern            36.0   
2         2  39.790760 -84.241550          0.01  US/Eastern            36.0   
3         2  39.752174 -84.239950          0.00  US/Eastern            36.0   
4         2  39.740670 -84.184135          0.01  US/Eastern            37.4   

   Humidity(%)  Pressure(in)  Visibility(mi) Wind_Direction  ...  Station  \
0        100.0         29.65            10.0           Calm  ...        0   
1        100.0         29.67            10.0             SW  ...        0   
2         89.0         29.65            10.0             NW  ...        0   
3         89.0         29.65            10.0             NW  ...        0   
4         93.0         29.63            10.0            WSW  ...        0   

   Stop Traffic_Calming  Traffic_Signal  Turning_Loop  Sunrise

In [2]:
from collections import Counter

# Count the occurrences of each severity level
severity_counts = Counter(df['Severity'])

# Print the counts
print(severity_counts)

Counter({2: 3501354, 3: 812367, 4: 69781, 1: 46893})


In [3]:
# Select numeric columns
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns

# Select categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

Numeric columns: Index(['Severity', 'Start_Lat', 'Start_Lng', 'Distance(mi)', 'Temperature(F)',
       'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)',
       'Precipitation(in)', 'Amenity', 'Bump', 'Crossing', 'Give_Way',
       'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop',
       'Traffic_Calming', 'Traffic_Signal', 'Turning_Loop',
       'duration_accident', 'Start_Hour', 'Start_Day', 'Start_Month'],
      dtype='object')
Categorical columns: Index(['Timezone', 'Wind_Direction', 'Weather_Condition', 'Sunrise_Sunset'], dtype='object')


In [4]:
# Example using one-hot encoding
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Check the shape after encoding
print("Shape after encoding:", df.shape)

Shape after encoding: (4430395, 117)


In [5]:
print(df.columns)

Index(['Severity', 'Start_Lat', 'Start_Lng', 'Distance(mi)', 'Temperature(F)',
       'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)',
       'Precipitation(in)',
       ...
       'Weather_Condition_Thunder', 'Weather_Condition_Thunder / Wintry Mix',
       'Weather_Condition_Thunder in the Vicinity',
       'Weather_Condition_Thunderstorm',
       'Weather_Condition_Thunderstorms and Rain', 'Weather_Condition_Tornado',
       'Weather_Condition_Volcanic Ash', 'Weather_Condition_Widespread Dust',
       'Weather_Condition_Wintry Mix', 'Sunrise_Sunset_Night'],
      dtype='object', length=117)


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

In [7]:
# Define target variable (Severity) and features
X = df.drop(columns=['Severity'])
y = df['Severity']  # Target variable

In [8]:
# Split into training (75%) and temporary test+validation (25%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.25, random_state=42)

# Split temporary test+validation into validation (15%) and test (10%)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.4, random_state=42)  # 40% of 25% = 10%


In [9]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

# Define oversampling and undersampling strategies
smote = SMOTE(sampling_strategy={1: 200000, 4: 200000}, random_state=42)
undersampler = RandomUnderSampler(sampling_strategy={2: 600000, 3: 600000}, random_state=42)

# Combine SMOTE and undersampling in a pipeline
smote_undersample_pipeline = Pipeline(steps=[('smote', smote), ('undersample', undersampler)])

# Apply the pipeline to resample data
X_resampled, y_resampled = smote_undersample_pipeline.fit_resample(X_train, y_train)

# Check the new distribution
print("Custom resampled class distribution:", Counter(y_resampled))

Custom resampled class distribution: Counter({2: 600000, 3: 600000, 1: 200000, 4: 200000})


In [11]:
print("X_train shape:", X_resampled.shape)
print("y_train shape:", y_resampled.shape)

X_train shape: (1600000, 116)
y_train shape: (1600000,)


In [18]:
y_resampled = y_resampled -1

In [13]:
y_test = y_test-1
y_val = y_val -1

In [21]:
from sklearn.model_selection import train_test_split

# Create a random subset of the training data (e.g., 50k samples)
X_train_subset, _, y_train_subset, _ = train_test_split(X_resampled, y_resampled, train_size=50000, random_state=42)

In [22]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],
    'max_depth': [6, 8],
    'subsample': [0.8],
    'colsample_bytree': [0.8, 1.0]
}

# Initialize the model
xgb_model = XGBClassifier(random_state=42)

# Perform GridSearchCV on the 50k subset
grid_search_xgb = GridSearchCV(estimator=xgb_model, param_grid=param_grid, scoring='f1_weighted', cv=3, n_jobs=-1, verbose=1)
grid_search_xgb.fit(X_train_subset, y_train_subset)

# Get the best model with the best parameters
best_xgb = grid_search_xgb.best_estimator_
print("Best Parameters for XGB model:", grid_search_xgb.best_params_)

Fitting 3 folds for each of 16 candidates, totalling 48 fits
Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 8, 'n_estimators': 200, 'subsample': 0.8}


In [24]:
# Now train the model with the entire training data
best_xgb.fit(X_resampled, y_resampled)

# Evaluate the model on the validation or test set
validation_score_XGB = best_xgb.score(X_val, y_val)
print("Validation Score with Best Model of XGB:", validation_score_XGB)

Validation Score with Best Model: 0.7583570458002976


In [25]:
import lightgbm as lgb

# Define the parameter grid for LightGBM
param_grid_lgbm = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],
    'max_depth': [6, 8],
    'subsample': [0.8],
    'colsample_bytree': [0.8, 1.0]
}

# Initialize the LightGBM model
lgbm_model = lgb.LGBMClassifier(random_state=42)

# Perform GridSearchCV on the 50k subset
grid_search_lgbm = GridSearchCV(estimator=lgbm_model, param_grid=param_grid_lgbm, scoring='f1_weighted', cv=3, n_jobs=-1, verbose=1)
grid_search_lgbm.fit(X_train_subset, y_train_subset)

# Get the best model
best_lgbm = grid_search_lgbm.best_estimator_
print("Best Parameters for LGBM model:", grid_search_lgbm.best_params_)

Fitting 3 folds for each of 16 candidates, totalling 48 fits
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002444 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2192
[LightGBM] [Info] Number of data points in the train set: 50000, number of used features: 65
[LightGBM] [Info] Start training from score -2.080722
[LightGBM] [Info] Start training from score -0.970747
[LightGBM] [Info] Start training from score -0.984034
[LightGBM] [Info] Start training from score -2.099155
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best g

In [26]:
# Now train the model with the entire training data
best_lgbm.fit(X_resampled, y_resampled)

# Evaluate the model on the validation or test set
validation_score_LGBM = best_lgbm.score(X_val, y_val)
print("Validation Score with Best Model of LGBM:", validation_score_LGBM)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.084073 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2228
[LightGBM] [Info] Number of data points in the train set: 1600000, number of used features: 83
[LightGBM] [Info] Start training from score -2.079442
[LightGBM] [Info] Start training from score -0.980829
[LightGBM] [Info] Start training from score -0.980829
[LightGBM] [Info] Start training from score -2.079442
Validation Score with Best Model of LGBM: 0.7595382802730833


In [27]:
from catboost import CatBoostClassifier

# Define the parameter grid for CatBoost
param_grid_cat = {
    'iterations': [100, 200],
    'learning_rate': [0.01, 0.1],
    'depth': [6, 8]
}

# Initialize the CatBoost model
cat_model = CatBoostClassifier(random_state=42, silent=True)

# Perform GridSearchCV on the 50k subset
grid_search_cat = GridSearchCV(estimator=cat_model, param_grid=param_grid_cat, scoring='f1_weighted', cv=3, n_jobs=-1, verbose=1)
grid_search_cat.fit(X_train_subset, y_train_subset)

# Get the best model
best_cat = grid_search_cat.best_estimator_
print("Best Parameters for catboost:", grid_search_lgbm.best_params_)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best Parameters for catboost: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 8, 'n_estimators': 200, 'subsample': 0.8}


In [28]:
# Now train the model with the entire training data
best_cat.fit(X_resampled, y_resampled)

# Evaluate the model on the validation or test set
validation_score_cat = best_cat.score(X_val, y_val)
print("Validation Score with Best Model of CatBoost:", validation_score_cat)

Validation Score with Best Model of CatBoost: 0.7218290625813509


In [29]:
from sklearn.ensemble import VotingClassifier

# Create the voting classifier
voting_clf = VotingClassifier(
    estimators=[('xgb', best_xgb), ('lgbm', best_lgbm), ('cat', best_cat)],
    voting='soft'
)

# Fit the ensemble model on the full training data
voting_clf.fit(X_resampled, y_resampled)

# Evaluate the ensemble model on the validation set
ensemble_score = voting_clf.score(X_val, y_val)
print("Ensemble Model Validation Score:", ensemble_score)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.092444 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2228
[LightGBM] [Info] Number of data points in the train set: 1600000, number of used features: 83
[LightGBM] [Info] Start training from score -2.079442
[LightGBM] [Info] Start training from score -0.980829
[LightGBM] [Info] Start training from score -0.980829
[LightGBM] [Info] Start training from score -2.079442
Ensemble Model Validation Score: 0.7515239429456226


In [31]:
# Predict on the test data
y_pred = voting_clf.predict(X_test)

# Evaluation of the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Ensemble Model Accuracy: {accuracy}")
print("Classification Report:\n", classification_report(y_test, y_pred))

Ensemble Model Accuracy: 0.7509525099313832
Classification Report:
               precision    recall  f1-score   support

           0       0.23      0.57      0.33      4814
           1       0.96      0.73      0.83    350090
           2       0.47      0.88      0.62     81203
           3       0.22      0.37      0.27      6933

    accuracy                           0.75    443040
   macro avg       0.47      0.64      0.51    443040
weighted avg       0.85      0.75      0.78    443040

